# Skyntri — Model A: Facial Skin Concerns (EfficientNet, multi-label)

This notebook builds a **clean, de-duplicated, leakage-free** multi-label dataset for facial skin concerns, trains an **EfficientNet** classifier, evaluates it properly, and exports exactly the files the Skyntri backend loads.

**It has not been run.** Upload your data, then run it top to bottom on Colab (GPU: *Runtime → Change runtime type → T4 GPU*).

| Step | Section |
|---|---|
| 0 | Setup and configuration |
| 1 | Load one or more datasets into a single manifest (unified label schema) |
| 2 | Clean: unreadable or tiny files, exact and near duplicates, Roboflow augmented copies, label conflicts, quality |
| 3 | Optional face crop, resize, cache cleaned images |
| 4 | Dataset statistics |
| 5 | Grouped, multi-label-stratified train/val/test split (no leakage) |
| 6 | `tf.data` pipeline and augmentation |
| 7 | EfficientNet model with masked, weighted focal loss |
| 8 | Training (two phases: frozen head, then fine-tuning) |
| 9 | Threshold tuning on validation |
| 10 | Test evaluation: per-class metrics, ROC/PR, calibration, bootstrap CIs, subgroup, error gallery |
| 11 | Export for the backend (`.keras` + metadata JSON) |
| 12 | Prediction (same preprocessing as the backend) |

**Label schema.** The class names are identical to the backend's (`DATASET3_CLASSES`), so recommendations keep working:
`Acne, Blackheads, Dark Spots, Dry Skin, Eye bags, Normal Skin, Oily Skin, Pores, Skin Redness, Wrinkles`.

**Preprocessing contract with the backend.** RGB → bilinear resize to `IMG_SIZE` → float32 in **[0, 255]**. Do **not** divide by 255: Keras EfficientNet rescales inside the model.

## Recommended datasets (download links)

Check each licence on its page before use and record it in the dataset card (section 11).

| Use | Dataset | Link | Notes |
|---|---|---|---|
| **Primary** (same 10 labels) | Roboflow *Skin-Problem-MultiLabel* | https://universe.roboflow.com/yolov10-kuvz2/skin-problem-multilabel-di2c7 · https://universe.roboflow.com/annisa-nadila-nasya-frarx/skin-problem-multilabel-rkbfr | Export as **"Multi-Label Classification" (CSV)**. Licence shown under *Cite this project*. Contains Roboflow augmented copies (handled below). |
| Supplement (CC BY 4.0) | Roboflow *facial* | https://universe.roboflow.com/facial-zfrfm/facial-ngz7q | Dark circles, enlarged pores, pimples, wrinkles, hyperpigmentation. Map names in `LABEL_SYNONYMS`. |
| Supplement (CC BY 4.0) | Roboflow *skin_multilabel* | https://universe.roboflow.com/skin-condition-detection/skin_multilabel | Acne, eye bags, wrinkles. Other labels are unknown → masked. |
| Supplement (severity 0–5) | Kaggle *Facial Skin Analysis and Type Classification* | https://www.kaggle.com/datasets/killa92/facial-skin-analysis-and-type-classification | Per-concern severity scores; converted with `SEVERITY_POSITIVE_MIN`. Check licence. |
| Supplement (single-label folders) | Kaggle *Skin Issues* | https://www.kaggle.com/datasets/ahmedismaiil/skin-issues-version-2-dataset-balanced | One folder per class → other labels unknown (masked loss). |
| Acne only (research) | ACNE04 v2 labels | https://github.com/AIpourlapeau/acne04v2 | Research use; cite Wu et al., ICCV 2019. |

Why several sources? No single public, permissively licensed dataset covers all ten facial concerns with clean labels. This notebook merges sources and uses a **masked loss**: a label that a source does not annotate is treated as *unknown* (`-1`), not *absent*. Missing labels therefore do not teach the model false negatives.

## 0. Setup

In [ ]:
# Colab: mount Google Drive (your datasets and outputs live there).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

!pip -q install "tensorflow==2.20.0" "opencv-python-headless<5" scikit-learn pandas matplotlib seaborn tqdm

In [ ]:
import hashlib
import json
import math
import os
import random
import shutil
import time
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    hamming_loss,
    jaccard_score,
    multilabel_confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from tqdm.auto import tqdm

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ============================== CONFIGURATION ==============================
SEED = 42

# Where your raw datasets are (uploaded to Drive) and where outputs go.
DATA_ROOT = Path("/content/drive/MyDrive/skyntri/datasets/facial")
WORK_DIR = Path("/content/drive/MyDrive/skyntri/work/facial")
EXPORT_DIR = Path("/content/drive/MyDrive/skyntri/export")

# Model
BACKBONE = "EfficientNetB0"     # "EfficientNetB0" (224) or "EfficientNetB3" (300)
IMG_SIZE = 224 if BACKBONE == "EfficientNetB0" else 300
PRETRAINED_WEIGHTS = "imagenet"  # set to None only for offline smoke tests
BATCH_SIZE = 32
HEAD_EPOCHS = 8
FINE_TUNE_EPOCHS = 25
HEAD_LR = 1e-3
FINE_TUNE_LR = 3e-5
UNFREEZE_LAST_N_LAYERS = 80      # how much of the backbone to fine-tune
FOCAL_GAMMA = 2.0
LABEL_SMOOTHING = 0.0

# Versioning written into the export (shown in Skyntri history/reports)
MODEL_VERSION = f"facial-concerns-{BACKBONE.lower()}-v2"
DATASET_VERSION = time.strftime("%Y-%m-%d")

# Cleaning
MIN_SIDE = 96                     # drop images smaller than this
NEAR_DUP_HAMMING = 4              # dHash distance for near duplicates (0-64)
MIN_SHARPNESS = 8.0               # variance of Laplacian on 256px gray
DARK_LUMA, BRIGHT_LUMA = 30, 240  # mean luminance limits
FACE_CROP = False                 # True: crop to detected face (+margin) when one is found
FACE_MARGIN = 0.35
CACHE_SIDE = 384                  # cleaned images are stored at this max side

# Split
SPLIT = {"train": 0.70, "val": 0.15, "test": 0.15}

# Kaggle severity sources: score >= this counts as "present"
SEVERITY_POSITIVE_MIN = 2

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

for p in (WORK_DIR, EXPORT_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Unified label schema - MUST match backend DATASET3_CLASSES (same names, same order)
CLASSES = [
    "Acne", "Blackheads", "Dark Spots", "Dry Skin", "Eye bags",
    "Normal Skin", "Oily Skin", "Pores", "Skin Redness", "Wrinkles",
]
NUM_CLASSES = len(CLASSES)
CLASS_INDEX = {c: i for i, c in enumerate(CLASSES)}

# Map label names used by different datasets to the unified schema.
# Keys are compared lower-case with spaces/underscores/hyphens removed.
LABEL_SYNONYMS = {
    "acne": "Acne", "pimple": "Acne", "pimples": "Acne", "papules": "Acne", "pustules": "Acne",
    "whitehead": "Acne", "whiteheads": "Acne", "nodules": "Acne",
    "blackhead": "Blackheads", "blackheads": "Blackheads",
    "darkspots": "Dark Spots", "darkspot": "Dark Spots", "hyperpigmentation": "Dark Spots",
    "pigmentation": "Dark Spots", "acnemarks": "Dark Spots", "acnescars": "Dark Spots", "melasma": "Dark Spots",
    "dryskin": "Dry Skin", "dry": "Dry Skin", "dryness": "Dry Skin",
    "eyebags": "Eye bags", "eyebag": "Eye bags", "darkcircles": "Eye bags", "puffyeyes": "Eye bags",
    "normal": "Normal Skin", "normalskin": "Normal Skin", "clear": "Normal Skin", "healthy": "Normal Skin",
    "oily": "Oily Skin", "oilyskin": "Oily Skin", "shine": "Oily Skin",
    "pores": "Pores", "enlargedpores": "Pores", "openpores": "Pores", "largepores": "Pores",
    "redness": "Skin Redness", "skinredness": "Skin Redness", "rosacea": "Skin Redness", "erythema": "Skin Redness",
    "wrinkles": "Wrinkles", "wrinkle": "Wrinkles", "finelines": "Wrinkles", "agingsigns": "Wrinkles",
}


def canon_label(name):
    key = "".join(ch for ch in str(name).lower() if ch.isalnum())
    return LABEL_SYNONYMS.get(key)


print("Classes:", CLASSES)

## 1. Load datasets into one manifest

Each loader returns rows `{path, source, group, y}` where `y` is a vector of length `NUM_CLASSES` with **1 = present, 0 = absent, -1 = unknown** (not annotated by that source).

Register the datasets you actually uploaded in the `SOURCES` list. Folder layouts:
* **Roboflow multi-label CSV export:** `train/ valid/ test/`, each with images and `_classes.csv` (`filename, <class1>, <class2>, ...`). Roboflow's own split is ignored; we re-split after de-duplication.
* **Severity CSV:** one CSV with an image column and one score column per concern.
* **Folder per class:** `root/<class name>/*.jpg` (single-label).

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def roboflow_group_key(filename):
    """Roboflow names exports like  img_12_jpg.rf.<hash>.jpg ; all copies of one
    original image (including Roboflow's augmented copies) share the stem."""
    name = Path(filename).name
    stem = name.split(".rf.")[0] if ".rf." in name else Path(name).stem
    return stem.lower()


def load_roboflow_multilabel(root, source):
    rows = []
    root = Path(root)
    for csv_path in sorted(root.rglob("_classes.csv")):
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]
        file_col = df.columns[0]
        label_cols = {c: canon_label(c) for c in df.columns[1:]}
        unmapped = [c for c, m in label_cols.items() if m is None]
        if unmapped:
            print(f"[{source}] ignoring unmapped columns: {unmapped}")
        annotated = {m for m in label_cols.values() if m}
        for _, r in df.iterrows():
            y = np.full(NUM_CLASSES, -1, dtype=np.int8)
            for cls in annotated:
                y[CLASS_INDEX[cls]] = 0
            for col, cls in label_cols.items():
                if cls and int(r[col]) == 1:
                    y[CLASS_INDEX[cls]] = 1
            path = csv_path.parent / str(r[file_col])
            rows.append({"path": str(path), "source": source,
                         "group": f"{source}:{roboflow_group_key(path.name)}", "y": y})
    return rows


def load_severity_csv(csv_path, image_dir, image_col, score_cols, source):
    """score_cols: {"csv column": "Unified class"}; scores >= SEVERITY_POSITIVE_MIN -> present."""
    df = pd.read_csv(csv_path)
    rows = []
    for _, r in df.iterrows():
        y = np.full(NUM_CLASSES, -1, dtype=np.int8)
        for col, cls in score_cols.items():
            if pd.notna(r[col]):
                y[CLASS_INDEX[cls]] = 1 if float(r[col]) >= SEVERITY_POSITIVE_MIN else 0
        path = Path(image_dir) / str(r[image_col])
        rows.append({"path": str(path), "source": source, "group": f"{source}:{path.stem.lower()}", "y": y})
    return rows


def load_folder_per_class(root, source):
    rows = []
    for class_dir in sorted(Path(root).iterdir()):
        if not class_dir.is_dir():
            continue
        cls = canon_label(class_dir.name)
        if cls is None:
            print(f"[{source}] skipping folder {class_dir.name!r} (no mapping)")
            continue
        for p in class_dir.rglob("*"):
            if p.suffix.lower() in IMG_EXT:
                y = np.full(NUM_CLASSES, -1, dtype=np.int8)
                y[CLASS_INDEX[cls]] = 1
                if cls == "Normal Skin":       # a "normal" image has no concerns
                    y[:] = 0
                    y[CLASS_INDEX["Normal Skin"]] = 1
                rows.append({"path": str(p), "source": source,
                             "group": f"{source}:{roboflow_group_key(p.name)}", "y": y})
    return rows


# ---- Register the datasets you uploaded (comment out the ones you do not use) ----
SOURCES = [
    lambda: load_roboflow_multilabel(DATA_ROOT / "skin_problem_multilabel", "rf_skin_problem_multilabel"),
    # lambda: load_roboflow_multilabel(DATA_ROOT / "rf_facial", "rf_facial"),
    # lambda: load_roboflow_multilabel(DATA_ROOT / "rf_skin_multilabel", "rf_skin_multilabel"),
    # lambda: load_folder_per_class(DATA_ROOT / "kaggle_skin_issues", "kaggle_skin_issues"),
    # lambda: load_severity_csv(
    #     DATA_ROOT / "kaggle_facial_severity/labels.csv", DATA_ROOT / "kaggle_facial_severity/images",
    #     image_col="image", source="kaggle_facial_severity",
    #     score_cols={"acne": "Acne", "dark_spots": "Dark Spots", "pores": "Pores",
    #                 "wrinkles": "Wrinkles", "redness": "Skin Redness"}),
]

rows = []
for loader in SOURCES:
    loaded = loader()
    print(f"loaded {len(loaded):6d} rows")
    rows.extend(loaded)

manifest = pd.DataFrame(rows)
print("Total rows:", len(manifest))
manifest.head()

## 2. Cleaning

1. **Unreadable / tiny / non-RGB-decodable** files are removed.
2. **Exact duplicates** (same MD5) are merged and their labels combined.
3. **Near duplicates** (perceptual dHash within `NEAR_DUP_HAMMING` bits), which include Roboflow's flipped/brightened copies, are put in the same **group**, so they always land in the same split. This prevents test-set leakage, which inflates metrics.
4. **Label conflicts:** "Normal Skin" together with any concern → "Normal Skin" is set to unknown. Rows with no positive label and no annotated negatives are dropped.
5. **Quality:** too dark, over-exposed or very blurry photos are removed. These are the same kinds of photos the app's input gate rejects.

In [ ]:
def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def dhash(img_gray, size=8):
    small = cv2.resize(img_gray, (size + 1, size), interpolation=cv2.INTER_AREA)
    diff = small[:, 1:] > small[:, :-1]
    return np.packbits(diff.flatten())  # 64 bits -> 8 bytes


def quality_stats(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    g = cv2.resize(gray, (256, 256), interpolation=cv2.INTER_AREA)
    return {
        "luma": float(g.mean()),
        "contrast": float(g.std()),
        "sharpness": float(cv2.Laplacian(g, cv2.CV_64F).var()),
    }


def inspect(path):
    try:
        with Image.open(path) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            rgb = np.asarray(im)
    except Exception as exc:
        return {"ok": False, "reason": f"unreadable: {type(exc).__name__}"}
    h, w = rgb.shape[:2]
    if min(h, w) < MIN_SIDE:
        return {"ok": False, "reason": "too_small"}
    q = quality_stats(rgb)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return {"ok": True, "w": w, "h": h, "md5": file_md5(path), "dhash": dhash(gray).tobytes().hex(), **q}


info = [inspect(p) for p in tqdm(manifest["path"], desc="inspecting")]
info = pd.DataFrame(info)
if "reason" not in info:
    info["reason"] = None
manifest = pd.concat([manifest.reset_index(drop=True), info], axis=1)

print(manifest["reason"].value_counts(dropna=True))
manifest = manifest[manifest["ok"] == True].copy()  # noqa: E712

# ---- quality filter ----
bad_quality = (
    (manifest["luma"] < DARK_LUMA) | (manifest["luma"] > BRIGHT_LUMA) | (manifest["sharpness"] < MIN_SHARPNESS)
)
print("Removed for quality:", int(bad_quality.sum()))
manifest = manifest[~bad_quality].copy()

In [ ]:
# ---- exact duplicates: merge labels ----
def merge_labels(ys):
    ys = np.stack(ys)
    merged = np.full(NUM_CLASSES, -1, dtype=np.int8)
    for k in range(NUM_CLASSES):
        col = ys[:, k]
        if (col == 1).any():
            merged[k] = 1
        elif (col == 0).any():
            merged[k] = 0
    return merged


before = len(manifest)
agg = []
for md5, grp in manifest.groupby("md5"):
    first = grp.iloc[0].to_dict()
    first["y"] = merge_labels(list(grp["y"]))
    first["group"] = grp["group"].iloc[0]
    agg.append(first)
manifest = pd.DataFrame(agg)
print(f"Exact duplicates merged: {before - len(manifest)}")

# ---- near duplicates: union-find over dHash (bucketed by 16-bit prefix + full scan per bucket) ----
parent = {g: g for g in manifest["group"]}


def find(g):
    while parent[g] != g:
        parent[g] = parent[parent[g]]
        g = parent[g]
    return g


def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra


hashes = np.array([np.frombuffer(bytes.fromhex(h), dtype=np.uint8) for h in manifest["dhash"]])
bits = np.unpackbits(hashes, axis=1)  # (N, 64)
groups = manifest["group"].tolist()
# Multi-probe buckets: 4 bands of 16 bits; near duplicates share at least one band (pigeonhole for <=3 bits).
for band in range(4):
    keys = [bytes(row[band * 2: band * 2 + 2]) for row in hashes]
    buckets = defaultdict(list)
    for i, k in enumerate(keys):
        buckets[k].append(i)
    for idx in buckets.values():
        if len(idx) < 2:
            continue
        sub = bits[idx]
        for a in range(len(idx) - 1):          # row-by-row keeps memory small for big buckets
            dist = (sub[a + 1:] != sub[a]).sum(1)
            for off in np.nonzero(dist <= NEAR_DUP_HAMMING)[0]:
                union(groups[idx[a]], groups[idx[a + 1 + off]])

manifest["group"] = [find(g) for g in groups]
print("Images:", len(manifest), "| independent groups:", manifest["group"].nunique())

In [ ]:
# ---- label conflicts and empty rows ----
def fix_labels(y):
    y = y.copy()
    n = CLASS_INDEX["Normal Skin"]
    concerns = [i for i in range(NUM_CLASSES) if i != n]
    if y[n] == 1 and (y[concerns] == 1).any():
        y[n] = -1                       # contradictory -> unknown
    if (y[concerns] == 1).any() and y[n] == -1:
        y[n] = 0                        # a concern is present, so not "normal"
    return y


manifest["y"] = manifest["y"].apply(fix_labels)
has_info = manifest["y"].apply(lambda y: (y == 1).any())
print("Dropped rows without any positive label:", int((~has_info).sum()))
manifest = manifest[has_info].reset_index(drop=True)

Y = np.stack(manifest["y"].values)
print("Unknown label share per class:")
print(pd.Series((Y == -1).mean(0), index=CLASSES).round(3))

## 3. Optional face crop, resize and cache

In [ ]:
CACHE_DIR = WORK_DIR / "cleaned"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
face_detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


def crop_face(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    faces = face_detector.detectMultiScale(gray, 1.1, 5, minSize=(60, 60))
    if len(faces) == 0:
        return rgb, False
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    m = int(FACE_MARGIN * max(w, h))
    H, W = rgb.shape[:2]
    return rgb[max(0, y - m): min(H, y + h + m), max(0, x - m): min(W, x + w + m)], True


def cache_image(row):
    out = CACHE_DIR / f"{row['md5']}.jpg"
    cropped = False
    if not out.exists():
        with Image.open(row["path"]) as im:
            rgb = np.asarray(ImageOps.exif_transpose(im).convert("RGB"))
        if FACE_CROP:
            rgb, cropped = crop_face(rgb)
        im = Image.fromarray(rgb)
        im.thumbnail((CACHE_SIDE, CACHE_SIDE), Image.Resampling.LANCZOS)
        im.save(out, quality=95)
    return str(out), cropped


results = [cache_image(r) for r in tqdm(manifest.to_dict("records"), desc="caching")]
manifest["clean_path"] = [r[0] for r in results]
manifest["face_cropped"] = [r[1] for r in results]
manifest.to_pickle(WORK_DIR / "manifest_clean.pkl")
print("Cached", len(manifest), "images to", CACHE_DIR)

## 4. Dataset statistics

In [ ]:
Y = np.stack(manifest["y"].values)
stats = pd.DataFrame({
    "positives": (Y == 1).sum(0),
    "negatives": (Y == 0).sum(0),
    "unknown": (Y == -1).sum(0),
}, index=CLASSES)
stats["pos_rate_known"] = (stats["positives"] / (stats["positives"] + stats["negatives"]).clip(lower=1)).round(3)
display(stats)
display(manifest.groupby("source").size().rename("images"))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
stats["positives"].plot.bar(ax=axes[0], title="Positive examples per class")
P = (Y == 1).astype(int)
co = P.T @ P
import seaborn as sns
sns.heatmap(pd.DataFrame(co, index=CLASSES, columns=CLASSES), annot=True, fmt="d", ax=axes[1], cmap="Blues")
axes[1].set_title("Label co-occurrence")
plt.tight_layout(); plt.show()

print("Labels per image:", pd.Series(P.sum(1)).value_counts().sort_index().to_dict())

In [ ]:
# Visual check of random cleaned samples with labels
sample = manifest.sample(min(16, len(manifest)), random_state=SEED)
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    ax.imshow(Image.open(r["clean_path"]))
    ax.set_title(", ".join(CLASSES[i] for i in np.where(r["y"] == 1)[0]) or "-", fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Grouped, multi-label-stratified split

Whole **groups** (an original image with its near-duplicates and augmented copies) are assigned to one split. A greedy iterative stratification places groups so that each split gets its share of every class. Rare classes are placed first.

In [ ]:
def grouped_iterative_split(df, fractions, seed=SEED):
    rng = np.random.default_rng(seed)
    names = list(fractions)
    g = df.groupby("group")["y"].apply(lambda ys: (np.stack(ys.values) == 1).sum(0)).to_dict()
    sizes = df.groupby("group").size().to_dict()
    group_ids = list(g)
    rng.shuffle(group_ids)
    total_pos = np.sum([g[k] for k in group_ids], axis=0).astype(float)
    total_n = float(sum(sizes.values()))
    need_pos = {s: total_pos * fractions[s] for s in names}
    need_n = {s: total_n * fractions[s] for s in names}
    assign = {}
    # rarest class first
    rarity = np.argsort(total_pos)
    remaining = set(group_ids)
    for k in rarity:
        for gid in [x for x in group_ids if x in remaining and g[x][k] > 0]:
            best = max(names, key=lambda s: (need_pos[s][k], need_n[s]))
            assign[gid] = best
            need_pos[best] -= g[gid]
            need_n[best] -= sizes[gid]
            remaining.discard(gid)
    for gid in [x for x in group_ids if x in remaining]:
        best = max(names, key=lambda s: need_n[s])
        assign[gid] = best
        need_n[best] -= sizes[gid]
    return df["group"].map(assign)


manifest["split"] = grouped_iterative_split(manifest, SPLIT)
assert manifest.groupby("group")["split"].nunique().max() == 1, "leakage: a group spans splits"

split_stats = pd.DataFrame({
    s: (np.stack(manifest.loc[manifest.split == s, "y"].values) == 1).sum(0) for s in SPLIT
}, index=CLASSES)
split_stats.loc["images"] = manifest["split"].value_counts()
display(split_stats)
manifest.to_csv(WORK_DIR / "manifest_split.csv", index=False)

## 6. `tf.data` pipeline and augmentation

Augmentations are mild and **colour-safe**: no hue shift and only small brightness/contrast changes, because colour carries the label for *Skin Redness* and *Dark Spots*. Validation and test images are never augmented.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.04, fill_mode="reflect"),
    tf.keras.layers.RandomZoom((-0.10, 0.10), fill_mode="reflect"),
    tf.keras.layers.RandomTranslation(0.05, 0.05, fill_mode="reflect"),
    tf.keras.layers.RandomContrast(0.12),
    tf.keras.layers.RandomBrightness(0.08, value_range=(0, 255)),
], name="augment")


def load_image(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method="bilinear")   # same as backend
    return tf.cast(img, tf.float32)                                        # 0..255, no /255


def make_ds(df, training):
    paths = df["clean_path"].values
    ys = np.stack(df["y"].values).astype(np.float32)          # -1 = unknown (masked)
    ds = tf.data.Dataset.from_tensor_slices((paths, ys))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: (load_image(p), y), num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)
    if training:  # augment whole batches (faster, and the layers always see 4-D input)
        ds = ds.map(lambda x, y: (tf.clip_by_value(augment(x, training=True), 0, 255), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_df = manifest[manifest.split == "train"].reset_index(drop=True)
val_df = manifest[manifest.split == "val"].reset_index(drop=True)
test_df = manifest[manifest.split == "test"].reset_index(drop=True)
train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df, False), make_ds(test_df, False)
print(len(train_df), len(val_df), len(test_df))

In [ ]:
# Preview augmentations of one training image
x0 = load_image(train_df.loc[0, "clean_path"])
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img = x0 if i == 0 else tf.clip_by_value(augment(x0[None], training=True)[0], 0, 255)
    ax.imshow(tf.cast(img, tf.uint8)); ax.axis("off"); ax.set_title("original" if i == 0 else f"aug {i}")
plt.show()

## 7. Model and loss

* **Backbone:** Keras EfficientNet with ImageNet weights. Its rescaling layer is built in, so inputs stay 0..255.
* **Head:** GlobalAveragePooling → Dropout → Dense(`NUM_CLASSES`, sigmoid).
* **Loss:** *masked, class-weighted focal binary cross-entropy*. Unknown labels (`-1`) contribute nothing. Positive weights counter class imbalance.

In [ ]:
Y_train = np.stack(train_df["y"].values)
pos = (Y_train == 1).sum(0).astype(np.float32)
neg = (Y_train == 0).sum(0).astype(np.float32)
POS_WEIGHT = np.clip(np.sqrt(neg / np.maximum(pos, 1.0)), 1.0, 10.0).astype(np.float32)
print(pd.Series(POS_WEIGHT, index=CLASSES).round(2))


def masked_focal_bce(gamma=FOCAL_GAMMA, pos_weight=POS_WEIGHT, smoothing=LABEL_SMOOTHING):
    pw = tf.constant(pos_weight, tf.float32)

    def loss(y_true, y_pred):
        mask = tf.cast(y_true >= 0, tf.float32)
        y = tf.clip_by_value(y_true, 0.0, 1.0)
        y = y * (1.0 - smoothing) + 0.5 * smoothing
        p = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce = -(pw * y * tf.math.log(p) + (1.0 - y) * tf.math.log(1.0 - p))
        p_t = y * p + (1.0 - y) * (1.0 - p)
        focal = tf.pow(1.0 - p_t, gamma) * bce
        return tf.reduce_sum(focal * mask) / tf.maximum(tf.reduce_sum(mask), 1.0)

    return loss


def build_model():
    backbone_cls = getattr(tf.keras.applications, BACKBONE)
    base = backbone_cls(include_top=False, weights=PRETRAINED_WEIGHTS, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="sigmoid", name="concerns")(x)
    return tf.keras.Model(inputs, outputs, name=f"skyntri_facial_{BACKBONE.lower()}"), base


model, base = build_model()
model.summary(show_trainable=True)

In [ ]:
def predict_probs(m, ds):
    return np.concatenate([m.predict_on_batch(x) for x, _ in ds], axis=0)


def masked_macro_ap(y, p):
    aps = []
    for k in range(NUM_CLASSES):
        known = y[:, k] >= 0
        yk = y[known, k]
        if yk.min() != yk.max():
            aps.append(average_precision_score(yk, p[known, k]))
    return float(np.mean(aps)) if aps else float("nan")


class MaskedMacroAP(tf.keras.callbacks.Callback):
    """Validation macro PR-AUC that ignores unknown labels; used for checkpointing."""

    def __init__(self, ds, y):
        super().__init__()
        self.ds, self.y = ds, y

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        logs["val_macro_ap"] = masked_macro_ap(self.y, predict_probs(self.model, self.ds))
        print(f" - val_macro_ap: {logs['val_macro_ap']:.4f}")


Y_val = np.stack(val_df["y"].values)
CKPT = WORK_DIR / "best_facial.keras"


def callbacks(patience):
    return [
        MaskedMacroAP(val_ds, Y_val),
        tf.keras.callbacks.ModelCheckpoint(str(CKPT), monitor="val_macro_ap", mode="max", save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_macro_ap", mode="max", patience=patience, restore_best_weights=True),
        tf.keras.callbacks.CSVLogger(str(WORK_DIR / "training_log.csv"), append=True),
    ]

## 8. Training (run this yourself on Colab)

In [ ]:
# Phase 1: train the head with the backbone frozen
model.compile(optimizer=tf.keras.optimizers.AdamW(HEAD_LR, weight_decay=1e-4), loss=masked_focal_bce())
history_head = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, callbacks=callbacks(patience=3))

In [ ]:
# Phase 2: fine-tune the top of the backbone (BatchNorm layers stay frozen)
base.trainable = True
for layer in base.layers[:-UNFREEZE_LAST_N_LAYERS]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

steps = FINE_TUNE_EPOCHS * max(1, math.ceil(len(train_df) / BATCH_SIZE))
schedule = tf.keras.optimizers.schedules.CosineDecay(FINE_TUNE_LR, decay_steps=steps, alpha=0.05)
model.compile(optimizer=tf.keras.optimizers.AdamW(schedule, weight_decay=1e-5), loss=masked_focal_bce())
history_ft = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, callbacks=callbacks(patience=6))

log = pd.read_csv(WORK_DIR / "training_log.csv")
log[["loss", "val_loss"]].plot(title="Loss"); plt.show()
log[["val_macro_ap"]].plot(title="Validation macro PR-AUC"); plt.show()

## 9. Threshold tuning on the validation set

Each class gets its own threshold: the value that maximises F1 on **validation** data, subject to a minimum precision so a label is not shown too eagerly. The test set is never used here.

In [ ]:
best = tf.keras.models.load_model(CKPT, compile=False)
p_val = predict_probs(best, val_ds)

MIN_PRECISION = 0.40
thresholds = {}
for k, cls in enumerate(CLASSES):
    known = Y_val[:, k] >= 0
    yk, pk = Y_val[known, k], p_val[known, k]
    if yk.sum() == 0 or yk.min() == yk.max():
        thresholds[cls] = 0.5
        continue
    prec, rec, thr = precision_recall_curve(yk, pk)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-9)
    ok = prec[:-1] >= MIN_PRECISION
    idx = int(np.argmax(np.where(ok, f1[:-1], -1))) if ok.any() else int(np.argmax(f1[:-1]))
    thresholds[cls] = float(np.clip(thr[idx], 0.05, 0.95))

pd.Series(thresholds).round(3)

## 10. Test evaluation

In [ ]:
p_test = predict_probs(best, test_ds)
Y_test = np.stack(test_df["y"].values)
T = np.array([thresholds[c] for c in CLASSES])
pred = (p_test >= T).astype(int)

rows = []
for k, cls in enumerate(CLASSES):
    known = Y_test[:, k] >= 0
    yk, pk, dk = Y_test[known, k], p_test[known, k], pred[known, k]
    both = yk.min() != yk.max() if len(yk) else False
    tn = int(((dk == 0) & (yk == 0)).sum()); fp = int(((dk == 1) & (yk == 0)).sum())
    rows.append({
        "class": cls, "support_pos": int(yk.sum()), "support_known": int(len(yk)),
        "threshold": thresholds[cls],
        "precision": precision_score(yk, dk, zero_division=0),
        "recall/sensitivity": recall_score(yk, dk, zero_division=0),
        "specificity": tn / max(tn + fp, 1),
        "f1": f1_score(yk, dk, zero_division=0),
        "roc_auc": roc_auc_score(yk, pk) if both else np.nan,
        "pr_auc(AP)": average_precision_score(yk, pk) if both else np.nan,
    })
per_class = pd.DataFrame(rows).set_index("class")
display(per_class.round(3))

# Aggregate metrics on fully-labelled test rows only (rows from partially labelled
# sources are evaluated per class above, where unknown labels are excluded).
full = (Y_test >= 0).all(1)
if full.sum() < 20:
    print("WARNING: few fully-labelled test rows; aggregate metrics treat unknown labels as absent.")
    full = np.ones(len(Y_test), dtype=bool)
yf, pf = np.clip(Y_test[full], 0, 1), pred[full]
summary = {
    "test_images": int(len(Y_test)),
    "fully_labelled_test_images": int(full.sum()),
    "micro_f1": f1_score(yf, pf, average="micro", zero_division=0),
    "macro_f1": f1_score(yf, pf, average="macro", zero_division=0),
    "weighted_f1": f1_score(yf, pf, average="weighted", zero_division=0),
    "samples_jaccard": jaccard_score(yf, pf, average="samples", zero_division=0),
    "subset_accuracy": float((yf == pf).all(1).mean()),
    "hamming_loss": hamming_loss(yf, pf),
    "macro_roc_auc": float(np.nanmean(per_class["roc_auc"])),
    "macro_pr_auc": float(np.nanmean(per_class["pr_auc(AP)"])),
}
pd.Series(summary).round(4)

In [ ]:
# Multilabel confusion matrices (per class: [[TN, FP], [FN, TP]])
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
mcm = multilabel_confusion_matrix(yf, pf)
for ax, cls, m in zip(axes.flat, CLASSES, mcm):
    sns.heatmap(m, annot=True, fmt="d", cbar=False, ax=ax, cmap="Blues",
                xticklabels=["pred 0", "pred 1"], yticklabels=["true 0", "true 1"])
    ax.set_title(cls)
plt.tight_layout(); plt.show()

# ROC and PR curves
fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 6))
for k, cls in enumerate(CLASSES):
    known = Y_test[:, k] >= 0
    yk, pk = Y_test[known, k], p_test[known, k]
    if yk.min() == yk.max():
        continue
    fpr, tpr, _ = roc_curve(yk, pk); a1.plot(fpr, tpr, label=f"{cls} ({roc_auc_score(yk, pk):.2f})")
    pr, rc, _ = precision_recall_curve(yk, pk); a2.plot(rc, pr, label=f"{cls} ({average_precision_score(yk, pk):.2f})")
a1.plot([0, 1], [0, 1], "k--"); a1.set_title("ROC"); a1.set_xlabel("FPR"); a1.set_ylabel("TPR"); a1.legend(fontsize=8)
a2.set_title("Precision-Recall"); a2.set_xlabel("Recall"); a2.set_ylabel("Precision"); a2.legend(fontsize=8)
plt.show()

In [ ]:
# Calibration: expected calibration error (ECE) and reliability diagram
def ece(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    total, err = len(y), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p >= lo) & (p < hi) if hi < 1 else (p >= lo) & (p <= hi)
        if m.any():
            err += m.sum() / total * abs(y[m].mean() - p[m].mean())
    return err


cal = {}
plt.figure(figsize=(6, 6))
for k, cls in enumerate(CLASSES):
    known = Y_test[:, k] >= 0
    yk, pk = Y_test[known, k], p_test[known, k]
    if len(yk) == 0:
        continue
    cal[cls] = ece(yk, pk)
    edges = np.linspace(0, 1, 11); mids, accs = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (pk >= lo) & (pk < hi)
        if m.sum() >= 5:
            mids.append(pk[m].mean()); accs.append(yk[m].mean())
    plt.plot(mids, accs, marker="o", label=cls)
plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("predicted probability"); plt.ylabel("observed frequency")
plt.title("Reliability diagram"); plt.legend(fontsize=7); plt.show()
per_class["ece"] = pd.Series(cal)
per_class[["ece"]].round(3)

In [ ]:
# Bootstrap 95% confidence intervals (resampling test images)
def bootstrap_ci(fn, n=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(yf))
    vals = []
    for _ in range(n):
        s = rng.choice(idx, len(idx), replace=True)
        vals.append(fn(yf[s], pf[s]))
    return np.percentile(vals, [2.5, 97.5])


ci = {
    "micro_f1": bootstrap_ci(lambda a, b: f1_score(a, b, average="micro", zero_division=0)),
    "macro_f1": bootstrap_ci(lambda a, b: f1_score(a, b, average="macro", zero_division=0)),
    "subset_accuracy": bootstrap_ci(lambda a, b: float((a == b).all(1).mean())),
}
for k, (lo, hi) in ci.items():
    print(f"{k}: {summary[k]:.4f}  (95% CI {lo:.4f} - {hi:.4f})")

In [ ]:
# Subgroup check: approximate skin tone from Individual Typology Angle (ITA) of skin pixels.
# ITA is an approximation, not self-reported skin type - use it to spot gaps, not to certify fairness.
def ita_group(path):
    rgb = np.asarray(Image.open(path).convert("RGB").resize((128, 128)))
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L, b = lab[..., 0] * 100 / 255, lab[..., 2] - 128
    ycrcb = cv2.cvtColor(rgb, cv2.COLOR_RGB2YCrCb)
    mask = (ycrcb[..., 1] >= 133) & (ycrcb[..., 1] <= 180) & (ycrcb[..., 2] >= 75) & (ycrcb[..., 2] <= 130)
    if mask.mean() < 0.05:
        return "unknown"
    ita = np.degrees(np.arctan2(np.median(L[mask]) - 50, np.median(b[mask])))
    return "light (ITA>41)" if ita > 41 else "intermediate (10-41)" if ita > 10 else "dark (ITA<=10)"


test_df["tone_group"] = [ita_group(p) for p in tqdm(test_df["clean_path"], desc="ITA")]
sub = []
for g, idx in test_df.groupby("tone_group").groups.items():
    idx = np.array(list(idx))
    m = (Y_test[idx] >= 0).all(1)
    if m.sum() < 10:
        continue
    sub.append({"group": g, "n": int(m.sum()),
                "macro_f1": f1_score(Y_test[idx][m], pred[idx][m], average="macro", zero_division=0),
                "micro_f1": f1_score(Y_test[idx][m], pred[idx][m], average="micro", zero_division=0)})
display(pd.DataFrame(sub))

# Per-source breakdown
display(pd.DataFrame([
    {"source": s, "n": len(i), "micro_f1": f1_score(
        np.clip(Y_test[list(i)], 0, 1), pred[list(i)], average="micro", zero_division=0)}
    for s, i in test_df.groupby("source").groups.items()
]))

In [ ]:
# Error gallery: most confident false positives and false negatives
def show_errors(kind, n=8):
    items = []
    for i in range(len(test_df)):
        for k in range(NUM_CLASSES):
            if Y_test[i, k] < 0:
                continue
            if kind == "FP" and pred[i, k] == 1 and Y_test[i, k] == 0:
                items.append((p_test[i, k], i, k))
            if kind == "FN" and pred[i, k] == 0 and Y_test[i, k] == 1:
                items.append((1 - p_test[i, k], i, k))
    items = sorted(items, reverse=True)[:n]
    fig, axes = plt.subplots(1, max(1, len(items)), figsize=(3 * max(1, len(items)), 3.5))
    for ax, (_, i, k) in zip(np.atleast_1d(axes), items):
        ax.imshow(Image.open(test_df.loc[i, "clean_path"])); ax.axis("off")
        ax.set_title(f"{kind} {CLASSES[k]}\np={p_test[i, k]:.2f}", fontsize=8)
    plt.show()


show_errors("FP"); show_errors("FN")

## 11. Export for the Skyntri backend

Files written to `EXPORT_DIR`:

| File | Put it in the repo at | Backend setting |
|---|---|---|
| `facial_concerns_efficientnet.keras` | `backend/models/` | `DATASET3_MODEL_PATH=models/facial_concerns_efficientnet.keras` |
| `facial_concerns_metadata.json` | `backend/models/` | `FACIAL_MODEL_METADATA_PATH=models/facial_concerns_metadata.json` |

The backend reads `classes`, `thresholds`, `image_size` and `model_version` from the metadata JSON. No code change is needed.

In [ ]:
MODEL_FILE = EXPORT_DIR / "facial_concerns_efficientnet.keras"
META_FILE = EXPORT_DIR / "facial_concerns_metadata.json"

best.save(MODEL_FILE)
sha256 = hashlib.sha256(MODEL_FILE.read_bytes()).hexdigest()

metadata = {
    "task": "facial_concerns",
    "model_version": MODEL_VERSION,
    "backbone": BACKBONE,
    "image_size": IMG_SIZE,
    "preprocessing": "RGB, bilinear resize to image_size, float32 0-255 (no /255; EfficientNet rescales internally)",
    "output": "sigmoid multi-label",
    "classes": CLASSES,
    "thresholds": thresholds,
    "sha256": sha256,
    "dataset_version": DATASET_VERSION,
    "sources": sorted(manifest["source"].unique().tolist()),
    "split_sizes": {s: int((manifest.split == s).sum()) for s in SPLIT},
    "test_metrics": {k: round(float(v), 4) for k, v in summary.items()},
    "test_metrics_ci95": {k: [round(float(a), 4), round(float(b), 4)] for k, (a, b) in ci.items()},
    "per_class": per_class.round(4).reset_index().to_dict(orient="records"),
    "notes": "AI estimate of visible facial skin-care concerns. Not a diagnosis.",
}
META_FILE.write_text(json.dumps(metadata, indent=2, default=float))
per_class.to_csv(EXPORT_DIR / "facial_concerns_per_class_metrics.csv")
print("Saved:", MODEL_FILE, META_FILE, "sha256:", sha256)

In [ ]:
# Dataset card (fill in the licence column after checking each source page)
card = ["# Dataset card - Skyntri facial concerns", "",
        f"Version: {DATASET_VERSION}", "", "| Source | Images | Licence | Consent/notes |", "|---|---|---|---|"]
for s, n in manifest["source"].value_counts().items():
    card.append(f"| {s} | {n} | TODO | TODO |")
card += ["", "Cleaning: unreadable/small removed, exact + near duplicates grouped, label conflicts resolved, "
         "quality filter (dark/bright/blur).", "Split: grouped iterative stratification 70/15/15 (no group spans splits)."]
(EXPORT_DIR / "facial_concerns_dataset_card.md").write_text("\n".join(card))

## 12. Prediction (identical to the backend)

This reproduces the backend path: RGB → bilinear resize → 0..255 float → model → per-class thresholds → "Normal Skin" is not a concern. If **no** label reaches its threshold, the result is *uncertain*, and the app asks for another photo.

In [ ]:
loaded = tf.keras.models.load_model(MODEL_FILE, compile=False)
meta = json.loads(META_FILE.read_text())


def preprocess_like_backend(path, size):
    with Image.open(path) as im:
        im = im.convert("RGB").resize((size, size), Image.Resampling.BILINEAR)
        return np.asarray(im, dtype=np.float32)[None]


def predict_concerns(path):
    x = preprocess_like_backend(path, meta["image_size"])
    probs = loaded.predict(x, verbose=0)[0]
    detected = [c for c, p in zip(meta["classes"], probs) if p >= meta["thresholds"][c]]
    if not detected:
        return {"status": "uncertain", "message": "We cannot confidently analyze this image."}
    concerns = [c for c in detected if c != "Normal Skin"]
    return {
        "status": "completed",
        "concerns": concerns or ["No supported facial concern detected"],
        "probabilities": {c: round(float(p), 3) for c, p in zip(meta["classes"], probs)},
        "model_version": meta["model_version"],
    }


# Sanity check: the reloaded model matches the trained one on a test batch
xb, _ = next(iter(test_ds))
assert np.allclose(best.predict_on_batch(xb), loaded.predict_on_batch(xb), atol=1e-5)

# Try it on any image (upload one in Colab with the file browser)
example = test_df.loc[0, "clean_path"]
predict_concerns(example)